# Phase 1: Data Preprocessing
This notebook explains my thought process behind the data preprocessing pipeline I implemented in `src/data/make_dataset.py` and `src/data/preprocessor.py`.

The goal of this phase is to turn raw tabular CSVs collected from Accelerometer & Gyroscope sensors into a structured dataset suitable for machine learning algorithms.


In [ ]:
import pandas as pd
import numpy as np
from glob import glob
import os
import re
import matplotlib.pyplot as plt

# The raw data lies here relative to the project structure
DATA_PATH = "../data/raw/MetaMotion"

## 1. Extracting Metadata from Filenames
The raw data contains multiple CSV files for different sensors, participants, exercises, and heaviness. All of this metadata is wonderfully encoded in the file name itself (e.g., `A-bench-heavy_MetaWear_2019-01-14T14.22.49.165_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv`).

I used a helper function to parse these strings so I can add them as separate columns in my DataFrame.


In [ ]:
def extract_metadata(filename):
    basename = os.path.basename(filename)
    parts = basename.split('-')
    
    participant = parts[0]
    label = parts[1]
    
    # E.g. heavy2_MetaWear_... -> heavy2 -> heavy
    category_with_digits = parts[2].split('_')[0]
    category = re.sub(r'\d+$', '', category_with_digits)
    
    return participant, label, category

print("Extracted:", extract_metadata("A-bench-heavy2_MetaWear_2019-01-11T16.10.08.270_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv"))


## 2. Temporal Alignment (`epoch` -> DatetimeIndex)
Sensor readings are captured at high frequency with an `epoch (ms)` timestamp. Transforming these into a Pandas `DatetimeIndex` simplifies subsequent time series operations (like resampling or rolling averages). 


In [ ]:
def process_timestamps(df):
    if "epoch (ms)" in df.columns:
        df.index = pd.to_datetime(df["epoch (ms)"], unit="ms")
        del df["epoch (ms)"]
    
    for col in ["time (01:00)", "elapsed (s)"]:
        if col in df.columns:
            del df[col]
            
    return df


## 3. The `DataLoader` Pipeline
`src/data/preprocessor.py` defines a `DataLoader` class to encapsulate reading raw CSV files, merging sensors, and temporal alignment.

Separating files into logical `sets` for accelerometer and gyroscope streams allows us to distinguish between independent workout bouts. The pipeline outputs a single merged DataFrame matching accelerometer and gyroscope metrics on their timestamps.


In [ ]:
# Simulated DataLoader logic snippet
print("DataLoader Pipeline involves:")
print("1. Iterating over all .csv files")
print("2. Mapping each file to a 'set' ID")
print("3. Converting 'epoch (ms)' to a datetime index")
print("4. pd.concat() merging acc & gyr columns horizontally")


## 4. Resampling
Data coming from raw sensors might have slightly varying frequencies or noise. Resampling standardizes the time steps. We define a 200ms window where numerical values are averaged (`mean`) and categorical values retain their last observed state (`last`).

To prevent overlapping edges, the data is sliced by day first before applying `.resample('200ms')`.


In [ ]:
# Example Resampling Rules
sampling_dict = {
    'acc_x': "mean", 'acc_y': "mean", 'acc_z': "mean",
    'gyr_x': "mean", 'gyr_y': "mean", 'gyr_z': "mean",
    'participant': "last", 'label': "last", 'category': "last", 'set': "last"
}

# Example usage (commented out):
# days = [g for n, g in final_dataset.groupby(pd.Grouper(freq='D'))]
# resampled = pd.concat([day.resample('200ms').apply(sampling_dict).dropna() for day in days])


## 5. Detecting & Removing Outliers
Real-world sensor data includes jarring movements or device glitches. The `OutlierRemover` class in `preprocessor.py` introduces 3 methods for anomaly detection:
1. **IQR**: Classic method checking bounds outside `1.5 * Interquartile Range`.
2. **Chauvenet's Criterion**: Evaluates probability of a data point appearing under a normal distribution.
3. **LOF (Local Outlier Factor)**: Machine Learning approach looking at local density compared to neighbors.

It's critical in fitness data that we can run these techniques **per label** (e.g. `per_label=True`). A motion that is an outlier for a bench press might be the standard motion for a deadlift!


In [ ]:
# Conceptual usage of OutlierRemover
# from src.data.preprocessor import OutlierRemover
# 
# dataset_cleaned = OutlierRemover.remove_outliers(
#     dataset, 
#     columns=["acc_x", "acc_y", "acc_z", ...], 
#     method="chauvenet", 
#     per_label=True
# )
